In [5]:
# filename: rune_value_normalization_v4.ipynb

import pandas as pd
import re
import numpy as np
import json

# === Load rune list from item_data.json ===
item_data_path = '/Users/buddy/Desktop/traderie/data/item_data.json'

with open(item_data_path, 'r') as f:
    item_data = json.load(f)

runes = list(item_data['Runes'].keys())

# === Load trade data ===
trade_data_path = '/Users/buddy/Desktop/traderie/data/normalized/normalized_trades_pc_sc_nl.csv'
df = pd.read_csv(trade_data_path)

# === Helper ===
def parse_quantity(s, rune):
    matches = re.findall(fr'{re.escape(rune)}:(\d+)', s)
    return int(matches[0]) if matches else 0

results = []

for rune in runes:
    if rune == 'Ist Rune':
        continue  # skip Ist itself

    single_item = df[df['NumAsks'] == 1]

    # Ist for Rune (bid side)
    ist_for_rune = single_item[
        (single_item['Offered'].str.contains('Ist Rune')) &
        (single_item['Requested'].str.contains(re.escape(rune)))
    ].copy()

    ist_for_rune['IstQty'] = ist_for_rune['Offered'].apply(lambda x: parse_quantity(x, 'Ist Rune'))
    ist_for_rune['RuneQty'] = ist_for_rune['Requested'].apply(lambda x: parse_quantity(x, rune))
    ist_for_rune['IstsPerRune'] = ist_for_rune['IstQty'] / ist_for_rune['RuneQty']

    # Rune for Ist (ask side)
    rune_for_ist = single_item[
        (single_item['Offered'].str.contains(re.escape(rune))) &
        (single_item['Requested'].str.contains('Ist Rune'))
    ].copy()

    rune_for_ist['IstQty'] = rune_for_ist['Requested'].apply(lambda x: parse_quantity(x, 'Ist Rune'))
    rune_for_ist['RuneQty'] = rune_for_ist['Offered'].apply(lambda x: parse_quantity(x, rune))
    rune_for_ist['IstsPerRune'] = rune_for_ist['IstQty'] / rune_for_ist['RuneQty']

    # Outlier filter (conservative wide band)
    ist_for_rune_filtered = ist_for_rune[(ist_for_rune['IstsPerRune'] >= 0.5) & (ist_for_rune['IstsPerRune'] <= 50)]
    rune_for_ist_filtered = rune_for_ist[(rune_for_ist['IstsPerRune'] >= 0.5) & (rune_for_ist['IstsPerRune'] <= 50)]

    # Split into small (≤2) and bulk (≥3)
    small_ist = ist_for_rune_filtered[ist_for_rune_filtered['RuneQty'] <= 2]
    bulk_ist = ist_for_rune_filtered[ist_for_rune_filtered['RuneQty'] >= 3]
    small_rune = rune_for_ist_filtered[rune_for_ist_filtered['RuneQty'] <= 2]
    bulk_rune = rune_for_ist_filtered[rune_for_ist_filtered['RuneQty'] >= 3]

    # VWAP calc (hidden behind the scenes)
    def vwap(df):
        rune_sum = df['RuneQty'].sum()
        return df['IstQty'].sum() / rune_sum if rune_sum != 0 else None

    vwap_small_ist = vwap(small_ist)
    vwap_bulk_ist = vwap(bulk_ist)
    vwap_small_rune = vwap(small_rune)
    vwap_bulk_rune = vwap(bulk_rune)

    # Blended values
    blended_small = None
    blended_bulk = None

    if vwap_small_ist and vwap_small_rune:
        blended_small = (vwap_small_ist + vwap_small_rune) / 2
    elif vwap_small_ist:
        blended_small = vwap_small_ist
    elif vwap_small_rune:
        blended_small = vwap_small_rune

    if vwap_bulk_ist and vwap_bulk_rune:
        blended_bulk = (vwap_bulk_ist + vwap_bulk_rune) / 2
    elif vwap_bulk_ist:
        blended_bulk = vwap_bulk_ist
    elif vwap_bulk_rune:
        blended_bulk = vwap_bulk_rune

    # Store results
    results.append({
        'Rune': rune.replace(' Rune',''),
        'Small_Bid': vwap_small_ist, 'Small_Bid_Count': len(small_ist),
        'Small_Ask': vwap_small_rune, 'Small_Ask_Count': len(small_rune),
        'Small_Blended': blended_small,
        'Bulk_Bid': vwap_bulk_ist, 'Bulk_Bid_Count': len(bulk_ist),
        'Bulk_Ask': vwap_bulk_rune, 'Bulk_Ask_Count': len(bulk_rune),
        'Bulk_Blended': blended_bulk
    })

# === Assemble final dataframe ===
results_df = pd.DataFrame(results)

results_df['Total_Trades'] = (
    results_df['Small_Bid_Count'].fillna(0) + 
    results_df['Small_Ask_Count'].fillna(0) +
    results_df['Bulk_Bid_Count'].fillna(0) + 
    results_df['Bulk_Ask_Count'].fillna(0)
)

results_df = results_df.sort_values(by='Total_Trades', ascending=False)

# === Final clean output ===
for _, row in results_df.iterrows():
    rune = row['Rune']
    print(f"\n==== {rune} ====")

    print("SMALL TRADES:")
    if pd.notna(row['Small_Bid']):
        print(f"Ist for {rune}: {row['Small_Bid']:.2f} Ists per {rune} (count: {int(row['Small_Bid_Count'])})")
    else:
        print("No Ist for Rune small trades")

    if pd.notna(row['Small_Ask']):
        print(f"{rune} for Ist: {row['Small_Ask']:.2f} Ists per {rune} (count: {int(row['Small_Ask_Count'])})")
    else:
        print("No Rune for Ist small trades")

    if pd.notna(row['Small_Blended']):
        print(f"Blended Fair Value: {row['Small_Blended']:.2f} Ists per {rune}")
    else:
        print("No blended small fair value")

    print("\nBULK TRADES:")
    if pd.notna(row['Bulk_Bid']):
        print(f"Ist for {rune}: {row['Bulk_Bid']:.2f} Ists per {rune} (count: {int(row['Bulk_Bid_Count'])})")
    else:
        print("No Ist for Rune bulk trades")

    if pd.notna(row['Bulk_Ask']):
        print(f"{rune} for Ist: {row['Bulk_Ask']:.2f} Ists per {rune} (count: {int(row['Bulk_Ask_Count'])})")
    else:
        print("No Rune for Ist bulk trades")

    if pd.notna(row['Bulk_Blended']):
        print(f"Blended Fair Value: {row['Bulk_Blended']:.2f} Ists per {rune}")
    else:
        print("No blended bulk fair value")

# Export CSV
results_df.to_csv('rune_fmv_full_v4.csv', index=False)


==== Jah ====
SMALL TRADES:
Ist for Jah: 9.61 Ists per Jah (count: 358)
Jah for Ist: 11.95 Ists per Jah (count: 62)
Blended Fair Value: 10.78 Ists per Jah

BULK TRADES:
Ist for Jah: 12.90 Ists per Jah (count: 70)
Jah for Ist: 11.43 Ists per Jah (count: 4)
Blended Fair Value: 12.16 Ists per Jah

==== Mal ====
SMALL TRADES:
Ist for Mal: 1.75 Ists per Mal (count: 3)
Mal for Ist: 0.98 Ists per Mal (count: 61)
Blended Fair Value: 1.37 Ists per Mal

BULK TRADES:
Ist for Mal: 0.72 Ists per Mal (count: 5)
Mal for Ist: 0.93 Ists per Mal (count: 31)
Blended Fair Value: 0.83 Ists per Mal

==== Ohm ====
SMALL TRADES:
Ist for Ohm: 3.17 Ists per Ohm (count: 24)
Ohm for Ist: 4.08 Ists per Ohm (count: 52)
Blended Fair Value: 3.62 Ists per Ohm

BULK TRADES:
No Ist for Rune bulk trades
Ohm for Ist: 1.00 Ists per Ohm (count: 1)
Blended Fair Value: 1.00 Ists per Ohm

==== Lo ====
SMALL TRADES:
Ist for Lo: 4.21 Ists per Lo (count: 46)
Lo for Ist: 5.58 Ists per Lo (count: 24)
Blended Fair Value: 4.90 Ists 